# 04 · M3: дообучение rubert-tiny2 (GPU)
*Среда выполнения → Сменить тип → T4 GPU.* Веса сохраняются на Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
REPO_DIR = '/content/prompt-injection-detector'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/HPAINH/prompt-injection-detector.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.environ['PIG_ROOT'] = '/content/drive/MyDrive/prompt-injection'
os.makedirs(os.environ['PIG_ROOT'], exist_ok=True)

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s', force=True)
from pig.config import load_config, set_seed
cfg = load_config()
set_seed(cfg['seed'])
print('PIG_ROOT =', os.environ['PIG_ROOT'])

In [ ]:
import torch
assert torch.cuda.is_available(), 'включите GPU'
from pig.pipeline import load_data, train_transformer
splits = load_data(cfg)
det, history = train_transformer(cfg, splits)
print('порог M3:', det.threshold)

## Кривые обучения
Проверка: loss на val не растёт к концу обучения.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
h = pd.DataFrame(history)
tr = h.dropna(subset=['loss']) if 'loss' in h else pd.DataFrame()
ev = h.dropna(subset=['eval_loss'])
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
if len(tr): ax[0].plot(tr['step'], tr['loss'], label='train')
ax[0].plot(ev['step'], ev['eval_loss'], 'o-', label='val'); ax[0].legend(); ax[0].set_title('loss')
ax[1].plot(ev['epoch'], ev['eval_f1'], 'o-'); ax[1].set_title('F1 на val')
fig.savefig(cfg.path('reports') / 'm3_training.png', dpi=160)
ev[['epoch', 'eval_loss', 'eval_f1']]

In [ ]:
from pig.eval.metrics import evaluate
test = splits['test']
evaluate(det.name, test, det.predict_proba(test['text'].tolist()), det.threshold)

## Публикация весов в HF Hub (приватно, опционально)

In [ ]:
from google.colab import userdata
from huggingface_hub import login, HfApi
login(userdata.get('HF_TOKEN'))
api = HfApi()
repo = f"{api.whoami()['name']}/pi-guard-rubert-tiny2"
api.create_repo(repo, private=True, exist_ok=True)
api.upload_folder(folder_path=str(cfg.path('models') / 'M3_transformer'), repo_id=repo, ignore_patterns=['checkpoints/*'])
print(repo)